In [2]:
# import modules
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

In [3]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

# Create hologram using known ground truth parameters

In [5]:
# ground truth parameters used to generate a hologram using scattering theory
# optical parameters
medium_index = 1.33
illum_wavelen = 0.660
illum_polarization = (0.56, 0.83)
detector = hp.detector_grid(shape=100, spacing=0.177)

# geometric parameters
N_1_TRUE = 1.59
N_2_TRUE = 1.59
R_1_TRUE = 0.65
R_2_TRUE = 0.65
# originally for spheres on top of each other, changed to less problematic case
X1_TRUE = 5
Y1_TRUE = 5
Z1_TRUE = 5
X2_TRUE = 4
Y2_TRUE = 4
Z2_TRUE = 5

# derived geometric parameters
Xg_TRUE = (X1_TRUE + X2_TRUE)/2
Yg_TRUE = (Y1_TRUE + Y2_TRUE)/2
Zg_TRUE = (Z1_TRUE + Z2_TRUE)/2
GAP = np.sqrt((X1_TRUE-X2_TRUE)**2+(Y1_TRUE-Y2_TRUE)**2+(Z1_TRUE-Z2_TRUE)**2)
# should use absolute value here?
THETA = np.arccos(abs(Z1_TRUE-Z2_TRUE)/GAP)
if (X1_TRUE-X2_TRUE) != 0:
    # should I be using absolute value?
    PHI = np.arctan((Y1_TRUE-Y2_TRUE)/(X1_TRUE-X2_TRUE))
elif (Y1_TRUE-Y2_TRUE) > 0:
    PHI = (np.pi)/2
elif (Y1_TRUE-Y2_TRUE) < 0:
    PHI = -np.pi/2
# both x and y are the same so phi is undefined -> set to 2pi
else:
    PHI = 2*np.pi
# adjust range from -pi to pi into 0 to 2pi
if PHI < 0:
    PHI = 2*np.pi + PHI
SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_testing_1_'

In [6]:
print(GAP)
print(THETA)
print(PHI)

1.4142135623730951
1.5707963267948966
0.7853981633974483


In [7]:
# create a hologram from two spheres, with parameters specified above
s1 = Sphere(center=(X1_TRUE, Y1_TRUE, Z1_TRUE), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(X2_TRUE, Y2_TRUE, Z2_TRUE), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo1 = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
# need to specify noise sd if priors not uniform, here use one from Caroline's fit used
# in mcmc fitting notebook
# specifying it in this way seems to lead to errors when loading fits as it appears as
# a coordinate instead of an attribute
holo1['noise_sd'] = 0.00862558

hp.show(holo1)

2025-04-30 14:18:39.144 python[51056:562881] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-30 14:18:39.144 python[51056:562881] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [ ]:
# create a hologram from two spheres, one above the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 3.5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_over = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_over['noise_sd'] = 0.00862558
hp.show(holo_over)

In [7]:
# create a hologram from two spheres, one next to the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 4, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_side['noise_sd'] = 0.00862558
hp.show(holo_side)

# Set parameters used in model fitting/ initialization best guesses

In [8]:
# info for modelling/ fitting

SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957
# not sure what this is and if I should be changing it or not
DIMER_Z_GUESS = 4.20

# Create model normally using independent theta and phi

In [23]:
def create_model(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return AlphaModel(scatterer, alpha=alpha)  


In [16]:
# try to fit hologram (adapted from Caroline's code)

dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
theta = prior.Uniform(0-0.1, np.pi+0.1, name="Theta")
phi = prior.Uniform(0-0.1, 2*np.pi + 0.1, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'theta': theta, 'phi': phi, 'gap': 0, 'alpha': alpha}
model =  create_model(step_2_parameters)
cma_fit_strategy = CmaStrategy(popsize=100, walker_initial_pos=model.generate_guess(scaling=0.5, n=100))
results2 = hp.fit(dimer_holo, model, strategy=cma_fit_strategy)
hp.save(SAVEPATH+'_nogap.h5', results2)

print('Independent angle fit completed')
print(results2.guess_parameters)
print(results2.parameters)

[dhcp-10-250-135-142.harvard.edu:47863] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/3321757696/sm_segment.dhcp-10-250-135-142.501.c5fe0000.0 could be created.
[dhcp-10-250-135-142.harvard.edu:47866] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/73728000/sm_segment.dhcp-10-250-135-142.501.4650000.0 could be created.
[dhcp-10-250-135-142.harvard.edu:47864] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/3006070784/sm_segment.dhcp-10-250-135-142.501.b32d0000.0 could be created.
[dhcp-10-250-135-142.harvard.edu:47870] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/81002496/sm_segment.dhcp-1

KeyboardInterrupt: 

# Try von Mises-Fisher distrubtion using joint angles prior (seems to have problems)

In [9]:
def create_model_angles(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    angles = parameters['angles']
    # problem with theta, phi when they are in a joint distribution
    # solution 1) somehow use just one object
    # angles = parameters['angles']
    # solution 2) somehow treat theta, phi as a vector object
    # theta, phi = parameters['angles']
    # solution 3) define priors that angles can be split up into
    #theta = parameters['angles'].theta
    #phi = parameters['angles'].phi
    gap = parameters['gap']
    alpha = parameters['alpha']
    theta, phi = angles
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return AlphaModel(scatterer, alpha=alpha)
"""
Ultimately we need to have acceess to concentration parameter and mean theta and phi
angles prior might not be the best way to do this given the seperation problem, I'll ask Vinny
instead could use a prior that has these parameters in it but have seperate theta and phi
get around the seperability problem and address it at the model stage (will have to modify
the model anyway). Problem with this is that we need to split off the zero case potentially,
ie) when there is no good mean theta and phi value but maybe k = 0 case can be defined to handle 
this as a specific special case (any angular input should have the same uniform pdf (unbounded though))
"""

"\nUltimately we need to have acceess to concentration parameter and mean theta and phi\nangles prior might not be the best way to do this given the seperation problem, I'll ask Vinny\ninstead could use a prior that has these parameters in it but have seperate theta and phi\nget around the seperability problem and address it at the model stage (will have to modify\nthe model anyway). Problem with this is that we need to split off the zero case potentially,\nie) when there is no good mean theta and phi value but maybe k = 0 case can be defined to handle \nthis as a specific special case (any angular input should have the same uniform pdf (unbounded though))\n"

In [30]:
# try to fit hologram using new prior object

dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
angles = prior.Angles(0.01,np.pi, 2*np.pi, name="Angles")
#theta = prior.Uniform(0-0.1, np.pi+0.1, name="Theta")
#phi = prior.Uniform(0-0.1, 2*np.pi + 0.1, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'angles': angles, 'gap': 0, 'alpha': alpha}
model =  create_model_angles(step_2_parameters)
cma_fit_strategy = CmaStrategy(popsize=100, walker_initial_pos=model.generate_guess(scaling=0.5, n=100))
results2 = hp.fit(dimer_holo, model, strategy=cma_fit_strategy)
hp.save(SAVEPATH+'_nogap.h5', results2)

print('von Mises-Fisher angles fit completed')
print(results2.guess_parameters)
print(results2.parameters)

TypeError: cannot unpack non-iterable Angles object

{'Phi': 3.8084479858970015, 'Theta': 3.1330819884262735, 'Z': 4.33783964841273, 'Alpha': 0.8373962779856561}
{'Phi': 3.141592653589793, 'Theta': 1.5707963267948966, 'Z': 4.2, 'Alpha': 0.8}


In [10]:
print(angles)

Angles(concentration_parameter=0.01, theta=3.141592653589793, phi=6.283185307179586, guess=(3.141592653589793, 6.283185307179586), name='Angles')


## Explore what different steps/ objects in the model creating process are

In [10]:
dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
theta = prior.Uniform(0-0.1, np.pi+0.1, name="Theta")
phi = prior.Uniform(0-0.1, 2*np.pi + 0.1, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'theta': theta, 'phi': phi, 'gap': 0, 'alpha': alpha}
model =  create_model(step_2_parameters)

In [11]:
s1_r = step_2_parameters['r_1']
s2_r = step_2_parameters['r_2']
s1_n = step_2_parameters['n_1']
s2_n = step_2_parameters['n_2']
center_x = step_2_parameters['x_g']
center_y = step_2_parameters['y_g']
center_z = step_2_parameters['z_g']
theta = step_2_parameters['theta']
phi = step_2_parameters['phi']
gap = step_2_parameters['gap']
alpha = step_2_parameters['alpha']
    
gap_center = np.array([center_x, center_y, center_z])
components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
s1_center = gap_center + (s1_r + gap/2) * components
s2_center = gap_center - (s2_r + gap/2) * components
        
scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                    Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)

In [34]:
print(gap_center)

[5.0 5.0 Uniform(lower_bound=2, upper_bound=10, guess=4.2, name='Z')]


In [35]:
print(s1_center)

[TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'cos'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=3.241592653589793, guess=1.5707963267948966, name='Theta'),)))), 0.6749016953839639)), 5.0))
 TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-

In [10]:
print(scatterer.parameters)

{'0:n': 1.5848484802283918, '0:r': 0.6749016953839639, '0:center': [TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'cos'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=3.241592653589793, guess=1.5707963267948966, name='Theta'),)))), 0.6749016953839639)), 5.0)), TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPr

In [36]:
for parameter in scatterer._parameters:
    print(parameter)
    #print(parameter.base_prior)

0:n
0:r
0:center
1:n
1:r
1:center


In [22]:
print(map_keys(model))

NameError: name 'map_keys' is not defined

In [23]:
# lnprior acesses the .lnprob method for each base prior that makes up the 
# transformed priors used in the scatterer object
for parameter in model._parameters:
    print(parameter)

Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi')
Uniform(lower_bound=-0.1, upper_bound=3.241592653589793, guess=1.5707963267948966, name='Theta')
Uniform(lower_bound=2, upper_bound=10, guess=4.2, name='Z')
Uniform(lower_bound=0.5, upper_bound=1.2, guess=0.8, name='Alpha')


# Test dummy parameter method where theta and phi are combined in KaiModel object

In [7]:
# define model creation using KaiModel object
def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return model.KaiModel(scatterer, alpha=alpha)

In [10]:
# create fit model using dummy phi and theta and then KaiModel object

dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
theta = prior.Theta(0.1, np.pi, name="Theta")
phi = prior.Phi(0.1, 2*np.pi, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'theta': theta, 'phi': phi, 'gap': 0, 'alpha': alpha}
model2 =  create_kaimodel(step_2_parameters)

In [22]:
# set new save path
SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_dummy_parameters_1_'

In [23]:
# now try to actually fit
cma_fit_strategy = CmaStrategy(popsize=100, walker_initial_pos=model.generate_guess(scaling=0.5, n=100))
results2 = hp.fit(dimer_holo, model2, strategy=cma_fit_strategy)
hp.save(SAVEPATH+'_nogap.h5', results2)

print('von Mises-Fisher angles fit completed')
print(results2.guess_parameters)
print(results2.parameters)

[dhcp-10-250-168-9.harvard.edu:98097] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-168-9.501/jf.0/1645412352/sm_segment.dhcp-10-250-168-9.501.62130000.0 could be created.
[dhcp-10-250-168-9.harvard.edu:98098] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-168-9.501/jf.0/944504832/sm_segment.dhcp-10-250-168-9.501.384c0000.0 could be created.
[dhcp-10-250-168-9.harvard.edu:98101] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-168-9.501/jf.0/78053376/sm_segment.dhcp-10-250-168-9.501.4a70000.0 could be created.
[dhcp-10-250-168-9.harvard.edu:98100] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-168-9.501/jf.0/2914844672/sm_segment.dhcp-10-250-168-9.501.adbd0

von Mises-Fisher angles fit completed
{'Phi': 6.283185307179586, 'Theta': 0.1, 'Z': 4.2, 'Alpha': 0.8}
{'Phi': 6.911822224208861, 'Theta': 0.009978916675434199, 'Z': 4.314670676417424, 'Alpha': 0.7953374380511938}


## Skip CMA for now and so hold off on implementing model.generate_guess

In [28]:
dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
r_1 = prior.BoundedGaussian(R_1_MEAN, R_1_SIGMA, lower_bound=0, upper_bound=1.0, name="r_1")
r_2 = prior.BoundedGaussian(R_2_MEAN, R_2_SIGMA, lower_bound=0, upper_bound=1.0, name="r_2")
n_1 = prior.BoundedGaussian(N_1_MEAN, N_1_SIGMA, lower_bound=0, upper_bound=1.7, name="n_1")
n_2 = prior.BoundedGaussian(N_2_MEAN, N_2_SIGMA, lower_bound=0, upper_bound=1.7, name="n_2")
x_g = prior.BoundedGaussian(x, SPACING, lower_bound=(x-5), 
                            upper_bound=(x+5), name="x_g")
y_g = prior.BoundedGaussian(y, SPACING, lower_bound=(y-5), 
                            upper_bound=(y+5), name="y_g")
z_g = prior.BoundedGaussian(Zg_TRUE, 1, lower_bound=0, upper_bound=50, name="z_g")
# note: k argument needs to be the same for both theta and phi
theta = prior.Theta(5, THETA, name="theta")
phi = prior.Phi(5, PHI, name="phi")
gap = prior.BoundedGaussian((GAP-R_1_MEAN-R_2_MEAN), 0.005, lower_bound=0, 
                            upper_bound=R_1_MEAN, name="gap")
alpha = prior.BoundedGaussian(0.8, 0.5, lower_bound=0.5, 
                            upper_bound=1.2, name="alpha") 
step_5_parameters = {'r_1': r_1, 'r_2': r_2, 'n_1': n_1, 'n_2': n_2,
                    'x_g': x_g, 'y_g': y_g, 'z_g': z_g,
                    'theta': theta, 'phi': phi, 'gap': gap, 'alpha': alpha}
model5 = create_kaimodel(step_5_parameters)

In [29]:
model5._parameters

[BoundedGaussian(mu=1.5848484802283918, sd=0.00027320846407879903, lower_bound=0, upper_bound=1.7, name='n_1'),
 BoundedGaussian(mu=0.6749016953839639, sd=0.00047233977835876834, lower_bound=0, upper_bound=1.0, name='r_1'),
 BoundedGaussian(mu=5.0, sd=0.177, lower_bound=0.0, upper_bound=10.0, name='x_g'),
 BoundedGaussian(mu=0.18043335128645344, sd=0.005, lower_bound=0, upper_bound=0.6749016953839639, name='gap'),
 Phi(mu=6.283185307179586, name='phi', sd=1),
 Theta(mu=0.0, name='theta', sd=1),
 BoundedGaussian(mu=5.0, sd=0.177, lower_bound=0.0, upper_bound=10.0, name='y_g'),
 BoundedGaussian(mu=4.25, sd=1, lower_bound=0, upper_bound=50, name='z_g'),
 BoundedGaussian(mu=1.601784237444771, sd=0.0003353994780766957, lower_bound=0, upper_bound=1.7, name='n_2'),
 BoundedGaussian(mu=0.6446649533295826, sd=0.000617682113114549, lower_bound=0, upper_bound=1.0, name='r_2'),
 BoundedGaussian(mu=0.8, sd=0.5, lower_bound=0.5, upper_bound=1.2, name='alpha')]

In [30]:
# now try to actually fit

# originally 50 walkers but start with 22 for speed
nwalkers = 22

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    for p in model5._parameters:
        means.append(p.mu + np.random.normal(0,p.sd*0.5))
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers, walker_initial_pos=initial_guess)

In [31]:
initial_guess

array([[1.58482189e+00, 6.74397711e-01, 5.02258820e+00, 1.84082718e-01,
        4.76920756e-01, 5.05633786e-02, 4.94211058e+00, 3.58482044e+00,
        1.60185542e+00, 6.44584655e-01, 1.01466952e+00],
       [1.58487678e+00, 6.74806170e-01, 5.11946234e+00, 1.82773491e-01,
        3.48904328e-01, 2.94953991e+00, 4.97403194e+00, 3.42098053e+00,
        1.60190484e+00, 6.44164167e-01, 1.07461236e+00],
       [1.58485728e+00, 6.74955127e-01, 5.07286885e+00, 1.82839967e-01,
        4.71261635e-02, 2.89156315e-01, 4.98070505e+00, 3.96750481e+00,
        1.60135080e+00, 6.44756087e-01, 4.40966891e-01],
       [1.58472634e+00, 6.75123541e-01, 4.95024865e+00, 1.78243116e-01,
        3.95508595e-01, 3.04075075e+00, 5.11896698e+00, 5.04338682e+00,
        1.60165962e+00, 6.43950978e-01, 1.15354351e+00],
       [1.58491862e+00, 6.74832482e-01, 4.88931243e+00, 1.81142246e-01,
        4.27472993e-01, 2.92281278e+00, 4.98410173e+00, 3.92843771e+00,
        1.60176329e+00, 6.44495863e-01, 1.11064191e+

In [27]:
SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_dummy_parameters_initial_conditions_6'

In [32]:
results5 = hp.sample(dimer_holo, model5, strategy=emcee_strategy)
hp.save(SAVEPATH+'_mcmc.h5', results5)

print('von Mises-Fisher angles fit completed')
print(results5.guess_parameters)
print(results5.parameters)

[dhcp-10-250-135-142.harvard.edu:50962] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/3362062336/sm_segment.dhcp-10-250-135-142.501.c8650000.0 could be created.
[dhcp-10-250-135-142.harvard.edu:50958] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/3176726528/sm_segment.dhcp-10-250-135-142.501.bd590000.0 could be created.
[dhcp-10-250-135-142.harvard.edu:50960] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/3726442496/sm_segment.dhcp-10-250-135-142.501.de1d0000.0 could be created.
[dhcp-10-250-135-142.harvard.edu:50959] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-135-142.501/jf.0/1258684416/sm_segment.d

von Mises-Fisher angles fit completed
{'n_1': 1.5848484802283918, 'r_1': 0.6749016953839639, 'x_g': 5.0, 'gap': 0.18043335128645344, 'phi': 6.283185307179586, 'theta': 0.0, 'y_g': 5.0, 'z_g': 4.25, 'n_2': 1.601784237444771, 'r_2': 0.6446649533295826, 'alpha': 0.8}
{'n_1': 1.584818085361399, 'r_1': 0.6749474167645072, 'x_g': 4.998827952310007, 'gap': 0.18150644790336784, 'phi': 4.982966564452573, 'theta': 3.1433661675545386, 'y_g': 4.999597677416779, 'z_g': 4.323308846045852, 'n_2': 1.601728379570047, 'r_2': 0.6440550474134065, 'alpha': 0.9675622208759851}


In [17]:
# return real fit values
means = []
for p in model5._parameters:
        means.append(p.mu)
print(means)

[1.5848484802283918, 0.6749016953839639, 5.0, 0.18043335128645344, 6.283185307179586, 0.0, 5.0, 4.25, 1.601784237444771, 0.6446649533295826, 0.8]


In [18]:
print(dimer_holo.noise_sd)

<xarray.DataArray 'noise_sd' ()>
array(0.00862558)
Coordinates:
    noise_sd  float64 0.008626


In [31]:
# need scipy 1.15 while we have 1.10 is this new version incompatible? -> yes incompatible with parrellel tempering
mu = np.array([-np.sqrt(0.5), -np.sqrt(0.5), 0])
vmf = stats.vonmises_fisher(mu, 5)

AttributeError: module 'scipy.stats' has no attribute 'vonmises_fisher'

In [39]:
-11.425662431912794%(2*np.pi)

1.140708182446378

# Test different geometries

## Test with spheres side by side

## Test with spheres very slightly offset from one above the other

# Test different concentration parameters

# Visualize fit results

In [14]:
# reload fit example
sampler_object = hp.load('/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_testing_1__nogap.h5')


In [24]:
# reload latest fit for analysis
results5 = hp.load('/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_dummy_parameters_initial_conditions_5_mcmc.h5')

NoMetadata: File without metadata detected. To load raw images, use hp.load_image()

In [18]:
# reload latest fit for analysis
results5 = hp.load_image('/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_dummy_parameters_initial_conditions_5_mcmc.h5',holo1)

TypeError: float() argument must be a string or a number, not 'HDF5StubImageFile'

In [26]:
# try loading one of Caroline's files
separate_path1 = '/n/manoharan/cmartin/Fits/depletion_holography/old_data/single_particle_fits/particle1_lm_fit.h5' 
new_path = '/Volumes/manoharan_lab/cmartin/Fits/depletion_holography/old_data/mcmc_fits/dimer_frame750_mcmc.h5'
Caroline_mcmc_fit = hp.load(new_path)
print(Caroline_mcmc_fit)
# seems like problem is with my saved files

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.99351717, 1.00936735, 1.0398761 , ..., 0.99996907, 0.98941159,
       1.03758787])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 27.26 36.64 13.81 25.66 ... 30.27 39.82 26.02 24.78
  * y        (flat) float64 14.51 38.41 11.33 26.02 ... 31.68 7.08 7.257 14.69
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[27.258, 14.514, 0], [36.638999999999996, 38.409, 0...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.70710678, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            [0.007725493123657364]
    original_dims:       {'x': [8.318999999999999, 8.495999999999999, 8.673, ..., model=AlphaModel(_dummy_scatterer=Spheres(scatterers=[Sphere(n=0, r=0, center=[0, 0, 0]), Sphere(n=0, r=0, center=[0, 0, 0])], warn=False), theory=Multisphere(niter=200, eps=1e-06, meth=1, qeps1=1e-05, 

In [8]:
# try loading using h5py
results_path = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_dummy_parameters_initial_conditions_5_mcmc.h5'
# open the file in read mode and then save some portions of it as variables,
# report what's in it and close the file again
with h5.File(results_path, 'r') as file:
    print("Keys:", file.keys())
    # so far just saving the following two parts of the results object
    samples = file['samples'][:]
    lnprobs = file['lnprobs'][:]
    # google suggested the following to iterate through the parts of h5 and see what they are
    def print_hdf5_item_structure(name, obj):
        print(name)
        if isinstance(obj, h5.Dataset):
            print("  Data type:", obj.dtype)
            print("  Shape:", obj.shape)
        elif isinstance(obj, h5.Group):
            print("  Type: Group")

    file.visititems(print_hdf5_item_structure)
print(samples[:,999])

Keys: <KeysViewHDF5 ['walker', 'chain', 'noise_sd', 'data', 'point', 'lnprobs', 'parameter', 'samples']>
chain
  Data type: >f4
  Shape: (1000,)
data
  Data type: float64
  Shape: (8000,)
lnprobs
  Data type: float64
  Shape: (22, 1000)
noise_sd
  Data type: float64
  Shape: ()
parameter
  Data type: object
  Shape: (11,)
point
  Data type: int64
  Shape: (8000,)
samples
  Data type: float64
  Shape: (22, 1000, 11)
walker
  Data type: >f4
  Shape: (22,)
[[ 1.58497804e+00  6.74183051e-01  5.00059261e+00  2.09513618e-01
   3.90367445e+01 -5.65269121e-03  4.99800412e+00  4.30068370e+00
   1.59982996e+00  6.46603054e-01  9.81653978e-01]
 [ 1.58485407e+00  6.74279087e-01  4.99863578e+00  2.11791356e-01
   4.04766543e+01 -6.30429015e-03  4.99909258e+00  4.30786365e+00
   1.59963347e+00  6.46799680e-01  9.79702065e-01]
 [ 1.58473870e+00  6.74365306e-01  4.99770866e+00  2.13686552e-01
   4.15347722e+01 -6.61579661e-03  4.99923469e+00  4.31416664e+00
   1.59947294e+00  6.46962066e-01  9.7906707

In [9]:
# try to load fit result object using hp.load code
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        # Kai added this to move noise_sd to correct place
        data.attrs['noise_sd'] = data.coords['noise_sd'].to_numpy()
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            # seems like noise_sd should be attribute not coordinate
            coordnames.remove('noise_sd') # added this
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)
'''
def _unserialize(cls, dataset):
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])

        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        return outlist
'''

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 22, chain: 1000, parameter: 11)
Coordinates:
    noise_sd   float64 0.008626
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.999 1.073 1.001 0.9861 ... 0.9821 1.009 0.9785
    lnprobs    (walker, chain) float64 -inf -1.374e+06 ... 1.835e+04 1.835e+04
    samples    (walker, chain, parameter) float64 1.585 0.6752 ... 0.6469 0.9847
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 22\nnsamples: 1000\nnpixels: 8000\nw...
    time:      1074.3808801174164
    _kwargs:   {}\n
<xarray.DataArray (flat: 8000)>
array([0.99897765, 1.07268187, 1.00101717, ..., 0.98206483, 1.00895219,
       0.97847535])
Coordinates:
  * flat     (flat) object MultiIndex
  * x      

"\ndef _unserialize(cls, dataset):\n        data = dataset.data\n        data.attrs = unpack_attrs(data.attrs)\n        if '_flat' in data.attrs.keys():\n            flats = np.array(data.attrs['_flat']).T\n            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]\n            codes = [[level.index(f) for f in flat]\n                     for level, flat in zip(levels, flats)]\n            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])\n            coordnames = list(data.coords)\n            coordnames.remove('point')\n            coords = {coord: data[coord] for coord in coordnames}\n            coords['flat'] = flat_index\n            data = xr.DataArray(data.values, dims=coordnames + ['flat'],\n                                coords=coords, attrs=data.attrs)\n        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)\n        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)\n        outlist = [data, model, strategy]\n     

In [10]:
# seems like don't need model at least for current analysis
# end up with noise_sd as a coordinate which is weird
print(return_variable)
samples = return_variable.samples[:,999]
lnprob = return_variable.lnprobs[5]
# can get rid of noise_sd coord using .reset_coords('noise_sd', drop = True)
print(samples.reset_coords('noise_sd',drop=True))
print(lnprob.reset_coords('noise_sd',drop=True))
burnt_samples = return_variable.burn_in(110).samples[:,889]

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.99897765, 1.07268187, 1.00101717, ..., 0.98206483, 1.00895219,
       0.97847535])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 13.27 4.956 7.965 9.204 ... 0.708 1.593 2.478 15.4
  * y        (flat) float64 13.1 9.204 10.62 7.965 ... 1.416 15.4 16.46 1.416
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[13.274999999999999, 13.097999999999999, 0], [4.955...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862558
    original_dims:       {'x': [0.0, 0.177, 0.354, 0.5309999999999999, 0.708,..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=22, nsamples=1000, npixels=8000, w

In [73]:
# can plot things fine even without dropping this additional coordinate, but probably
# worth dropping anyway since will likely mess up converting to pd array
plt.plot(lnprobs[5])

In [11]:
# set results equal to loaded fit for further analysis
results5 = return_variable

In [36]:
# try loading results object using 2nd half of code in hp.load function
# loads it in a form we don't want
with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    '''
            if '_source_class' in ds.attrs:
                _source_class = ds.attrs.pop('_source_class')
                pathtok = _source_class.split('.')
                cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
                ds.close()
                return_variable = cls._load(results_path)
    '''
            # Xarray defaults to lazy loading of datasets, but I my reading of
            # things is that we will probably generally prefer eager loading
            # since our data is generally fairly small but we do lots of
            # calculations.
    ds = ds.load()

            # loaded dataset potential contains multiple DataArrays. We
            # need to find out their names and loop through them to unpack
            # metadata
    data_vars = list(ds.data_vars.keys())
    #for var in data_vars:
        #ds[var].attrs = unpack_attrs(ds[var].attrs)

            # return either a single DataArray or a DataSet containing
            # multiple DataArrays.
    if len(data_vars)==1:
        return_variable = ds[data_vars[0]]
    else:
        return_variable = ds

In [37]:
print(return_variable)

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 22, chain: 1000, parameter: 11)
Coordinates:
    noise_sd   float64 0.008626
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.999 1.073 1.001 0.9861 ... 0.9821 1.009 0.9785
    lnprobs    (walker, chain) float64 -inf -1.374e+06 ... 1.835e+04 1.835e+04
    samples    (walker, chain, parameter) float64 1.585 0.6752 ... 0.6469 0.9847
Attributes:
    model:          !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sp...
    strategy:       !EmceeStrategy\nnwalkers: 22\nnsamples: 1000\nnpixels: 80...
    time:           1074.3808801174164
    _source_class:  holopy.inference.SamplingResult
    _kwargs:        {}\n


In [8]:
print(sampler_object)

FitResult(data=<xarray.DataArray 'data' (x: 100, y: 100, z: 1)>
array([[[0.99935616],
        [1.0055194 ],
        [0.99735586],
        ...,
        [1.00166742],
        [0.99761792],
        [0.999776  ]],

       [[1.00557803],
        [0.99713269],
        [0.99292671],
        ...,
        [1.00138606],
        [0.99955587],
        [0.99755626]],

       [[0.99769151],
        [0.99296794],
        [1.00628997],
        ...,
...
        ...,
        [1.0008593 ],
        [0.99972428],
        [0.99920373]],

       [[0.995864  ],
        [0.99818589],
        [1.00089133],
        ...,
        [0.99973353],
        [0.99914273],
        [1.00031235]],

       [[1.00062857],
        [0.99695996],
        [0.99556501],
        ...,
        [0.99909096],
        [1.00029971],
        [1.0008597 ]]])
Coordinates:
  * x        (x) float64 0.0 0.177 0.354 0.531 0.708 ... 16.99 17.17 17.35 17.52
  * y        (y) float64 0.0 0.177 0.354 0.531 0.708 ... 16.99 17.17 17.35 17.52
  * z    

In [21]:
samples = sampler_object.samples
print(samples[:,44])
pd_samples = samples[:,44].to_dataframe()
print(pd_samples)

<xarray.DataArray 'samples' (walker: 1, parameter: 4)>
array([[3.80844799, 3.13308199, 4.33783965, 0.83739628]])
Coordinates:
  * parameter  (parameter) object 'Phi' 'Theta' 'Z' 'Alpha'
Dimensions without coordinates: walker
                   samples
walker parameter          
0      Phi        3.808448
       Theta      3.133082
       Z          4.337840
       Alpha      0.837396


In [22]:
samples = results5.samples
print(samples[:,999])

<xarray.DataArray (walker: 22, parameter: 11)>
array([[ 1.58497804e+00,  6.74183051e-01,  5.00059261e+00,
         2.09513618e-01,  3.90367445e+01, -5.65269121e-03,
         4.99800412e+00,  4.30068370e+00,  1.59982996e+00,
         6.46603054e-01,  9.81653978e-01],
       [ 1.58485407e+00,  6.74279087e-01,  4.99863578e+00,
         2.11791356e-01,  4.04766543e+01, -6.30429015e-03,
         4.99909258e+00,  4.30786365e+00,  1.59963347e+00,
         6.46799680e-01,  9.79702065e-01],
       [ 1.58473870e+00,  6.74365306e-01,  4.99770866e+00,
         2.13686552e-01,  4.15347722e+01, -6.61579661e-03,
         4.99923469e+00,  4.31416664e+00,  1.59947294e+00,
         6.46962066e-01,  9.79067079e-01],
       [ 1.58481933e+00,  6.74310777e-01,  4.99913773e+00,
         2.11712145e-01,  3.98363972e+01, -5.96584420e-03,
         4.99886578e+00,  4.30917661e+00,  1.59963876e+00,
         6.46795298e-01,  9.81846447e-01],
       [ 1.58488542e+00,  6.74274559e-01,  4.99874830e+00,
         2.102

In [14]:
print(lnprobs)
print(len(lnprobs))

[[             -inf -1373789.14201895 -1197802.6711464  ...
     17771.14395815    17890.49306802    17890.49306802]
 [             -inf              -inf              -inf ...
     18172.36546266    18172.36546266    18172.36546266]
 [             -inf -1091136.37966079 -1049098.38086647 ...
     18239.84331076    18239.84331076    18239.84331076]
 ...
 [ -701646.08440291  -701646.08440291  -701646.08440291 ...
     18073.29243672    18110.15401464    18110.15401464]
 [-2533901.28526044 -2533901.28526044  -508029.44154199 ...
    -32378.34850124   -32378.34850124   -32378.34850124]
 [-1418736.67480187 -1418736.67480187 -1418736.67480187 ...
     18350.60207277    18350.60207277    18350.60207277]]
22


In [15]:
plt.plot(lnprobs[5])

## Clumsy way to get sns.pairplot working by converting sampling object samples to pandas

In [26]:
pd_samples = samples[:,999].to_dataframe(name = 'values')
print(pd_samples)
# need to reorder so each parameter is a different column
#sns.pairplot(pd_samples,vars=var_names)

                     values
walker parameter           
0      n_1         1.584978
       r_1         0.674183
       x_g         5.000593
       gap         0.209514
       phi        39.036745
...                     ...
21     y_g         4.995612
       z_g         4.308659
       n_2         1.599525
       r_2         0.646920
       alpha       0.984696

[242 rows x 1 columns]


In [27]:
var_names = []
for index in pd_samples.index:
    if index[0] == 0:
        var_names.append(index[1])
print(var_names)

['n_1', 'r_1', 'x_g', 'gap', 'phi', 'theta', 'y_g', 'z_g', 'n_2', 'r_2', 'alpha']


In [28]:
pd_samples2 = pd_samples.T
print(pd_samples2)

walker           0                                                      \
parameter       n_1       r_1       x_g       gap        phi     theta   
values     1.584978  0.674183  5.000593  0.209514  39.036745 -0.005653   

walker                                            ...        21            \
parameter       y_g       z_g      n_2       r_2  ...       r_1       x_g   
values     4.998004  4.300684  1.59983  0.646603  ...  0.674257  5.000459   

walker                                                                  \
parameter       gap        phi     theta       y_g       z_g       n_2   
values     0.213426  42.468949 -0.005537  4.995612  4.308659  1.599525   

walker                        
parameter      r_2     alpha  
values     0.64692  0.984696  

[1 rows x 242 columns]


In [29]:
df = pd_samples.reset_index()
print (df)

     walker parameter     values
0         0       n_1   1.584978
1         0       r_1   0.674183
2         0       x_g   5.000593
3         0       gap   0.209514
4         0       phi  39.036745
..      ...       ...        ...
237      21       y_g   4.995612
238      21       z_g   4.308659
239      21       n_2   1.599525
240      21       r_2   0.646920
241      21     alpha   0.984696

[242 rows x 3 columns]


In [30]:
df1 = df.pivot('walker','parameter','values')
print(df1)
df2 = df1.rename_axis(index=None,columns=None)
print(df2)

parameter     alpha       gap       n_1       n_2        phi       r_1  \
walker                                                                   
0          0.981654  0.209514  1.584978  1.599830  39.036745  0.674183   
1          0.979702  0.211791  1.584854  1.599633  40.476654  0.674279   
2          0.979067  0.213687  1.584739  1.599473  41.534772  0.674365   
3          0.981846  0.211712  1.584819  1.599639  39.836397  0.674311   
4          0.979169  0.210285  1.584885  1.599749  38.773923  0.674275   
5          0.984400  0.212596  1.584827  1.599583  41.224146  0.674278   
6          0.979692  0.210779  1.584866  1.599709  39.212855  0.674285   
7          0.703074  0.180430  1.584903  1.601562   6.067905  0.674993   
8          0.975882  0.210948  1.584884  1.599692  39.716038  0.674273   
9          0.969443  0.216523  1.584873  1.599261  47.847360  0.674195   
10         0.705136  0.180388  1.584903  1.601562   6.004362  0.674995   
11         0.981817  0.210482  1.58486

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_26357/3726504942.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  df1 = df.pivot('walker','parameter','values')


In [33]:
sns.pairplot(df2)

## Work on futher data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [16]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[1].sel(parameter='gap')
print(gaps_example)

AttributeError: 'numpy.ndarray' object has no attribute 'sel'

In [54]:
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [98]:
# plot pre-burn in
for i in range(11):
    plt.plot(results5.lnprobs[i])

In [12]:
# Use .burn_in() to chop off data before a specific sample number
burnt_results5 = results5.burn_in(110) #110 seems good for this specific fit
plt.plot(burnt_results5.lnprobs[5])
# to do more systematically could maybe calculate some sort of slope vs maximum slope cutoff?

In [100]:
# looking at the fits for all the different walkers it's clear that several don't converge well
for i in range(11):
    plt.plot(burnt_results5.lnprobs[i])

In [13]:
new_sample_length = len(burnt_results5.lnprobs[i])
print(burnt_results5.lnprobs[i][new_sample_length-1])

NameError: name 'i' is not defined

### Visualize Data Traces

In [48]:
# look at some traces of theta
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta'))

In [52]:
# look at some traces of phi
for i in range(4):
    plt.plot(samples[i].sel(parameter='phi'))

In [56]:
# look at some traces of gap
for i in range(4):
    plt.plot(samples[i].sel(parameter='gap'))

In [51]:
# look at some of the traces for walkers that don't converge
plt.plot(samples[7].sel(parameter='theta'))
plt.plot(samples[10].sel(parameter='theta'))

In [46]:
# look at theta that did converge and add pi to them
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta')+np.pi)

In [54]:
plt.plot(samples[7].sel(parameter='phi'))
plt.plot(samples[10].sel(parameter='phi'))

In [57]:
plt.plot(samples[7].sel(parameter='gap'))
plt.plot(samples[10].sel(parameter='gap'))

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [14]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > 0:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
    converged_samples = converged_samples_nan[1:]
print(converged_samples)

<xarray.DataArray (walker: 16, chain: 890, parameter: 11)>
array([[[1.58486963, 0.67476936, 4.99181943, ..., 1.60172554,
         0.64459203, 0.94132368],
        [1.58486963, 0.67476936, 4.99181943, ..., 1.60172554,
         0.64459203, 0.94132368],
        [1.58486963, 0.67476936, 4.99181943, ..., 1.60172554,
         0.64459203, 0.94132368],
        ...,
        [1.58501345, 0.67416062, 5.00098314, ..., 1.59990984,
         0.64652232, 0.98216429],
        [1.58497804, 0.67418305, 5.00059261, ..., 1.59982996,
         0.64660305, 0.98165398],
        [1.58497804, 0.67418305, 5.00059261, ..., 1.59982996,
         0.64660305, 0.98165398]],

       [[1.58487827, 0.67475916, 4.99341489, ..., 1.60174616,
         0.64457045, 0.9382447 ],
        [1.58487827, 0.67475916, 4.99341489, ..., 1.60174616,
         0.64457045, 0.9382447 ],
        [1.58487827, 0.67475916, 4.99341489, ..., 1.60174616,
         0.64457045, 0.9382447 ],
...
        [1.58480322, 0.67432616, 4.99812864, ..., 1.599583

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [111]:
# plot converged samples
converged_id = [0,1,2,3,4,5,6,8,9]
for id in converged_id:
    plt.plot(burnt_results5.lnprobs[id])

### Decimate data so just keep independent fits

In [15]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [16]:
# look at what corresponding trace looks like
plt.plot(converged_samples[1].sel(parameter='gap'))

In [29]:
# look at autocorrelation of gap data from different walkers
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [31]:
# look at trace of gap from different walkers
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [35]:
# look at autocorrelation of theta data from different walkers
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [36]:
# look at trace of theta from different walkers
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='theta'))

In [129]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [119]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1

In [20]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

[[199, 188, 168, 178, 195, 174, 186, 189, 194, 185, 163, 179, 173, 188, 183, 187], [347, 293, 311, 274, 294, 295, 303, 289, 360, 235, 353, 328, 317, 316, 275, 266], [177, 155, 421, 422, 377, 454, 313, 423, 135, 314, 447, 298, 182, 162, 407, 285], [757, 784, 774, 808, 842, 799, 775, 773, 752, 750, 887, 796, 777, 829, 768, 786], [802, 810, 792, 811, 759, 807, 764, 813, 782, 791, 814, 792, 835, 807, 810, 799], [333, 406, 340, 780, 362, 365, 322, 334, 356, 263, 333, 364, 560, 373, 323, 383], [160, 148, 92, 126, 129, 96, 150, 129, 145, 145, 98, 122, 151, 112, 151, 157], [189, 178, 162, 176, 185, 172, 174, 183, 181, 184, 155, 169, 167, 179, 177, 182], [749, 779, 763, 773, 793, 774, 769, 758, 749, 748, 762, 789, 765, 804, 762, 786], [754, 782, 766, 777, 798, 777, 772, 761, 751, 750, 766, 791, 767, 807, 765, 786], [562, 747, 719, 691, 471, 706, 460, 704, 439, 733, 733, 415, 826, 839, 450, 478]]
887


In [21]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

799


In [22]:
# now use correlation time to decimate the data
chain_number = len(converged_samples[0])
number_ind_samples_per_walker = int(np.ceil(chain_number/overall_correlation_time))
independent_samples =[]
for i in range(number_ind_samples_per_walker):
    index = (chain_number-1-overall_correlation_time*i)
    if i==0:
        independent_samples = converged_samples[:,index]
    else:
        independent_samples = xr.concat([independent_samples,(converged_samples[:,index])], 'walker')

In [23]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')
#print(independent_samples_pd)

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_79738/3190556392.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')


In [24]:
sns.pairplot(independent_samples_pd)

2025-04-24 15:20:56.187 python[79738:12409988] The class 'NSSavePanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


In [25]:
# seems like the parameter sets from the end of the run and the beginning of the run
# cluster in different ways, lets try seperating them

# get parameters at the end of the fitting process
end_of_run_ind = independent_samples[:16]
end_of_run_ind_pd = end_of_run_ind.to_dataframe(name = 'independent samples')
end_of_run_ind_pd = end_of_run_ind_pd.reset_index()
end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
print(end_of_run_ind_pd)

# get parameters from earlier on in the fitting process (~correlation time before the end)
early_run_ind = independent_samples[16:]
early_run_ind_pd = early_run_ind.to_dataframe(name = 'independent samples')
early_run_ind_pd = early_run_ind_pd.reset_index()
early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')
#print(early_run_ind_pd)

parameter     alpha       gap       n_1       n_2        phi       r_1  \
walker                                                                   
0          0.981654  0.209514  1.584978  1.599830  39.036745  0.674183   
1          0.979702  0.211791  1.584854  1.599633  40.476654  0.674279   
2          0.979067  0.213687  1.584739  1.599473  41.534772  0.674365   
3          0.981846  0.211712  1.584819  1.599639  39.836397  0.674311   
4          0.979169  0.210285  1.584885  1.599749  38.773923  0.674275   
5          0.984400  0.212596  1.584827  1.599583  41.224146  0.674278   
6          0.979692  0.210779  1.584866  1.599709  39.212855  0.674285   
7          0.975882  0.210948  1.584884  1.599692  39.716038  0.674273   
8          0.969443  0.216523  1.584873  1.599261  47.847360  0.674195   
9          0.981817  0.210482  1.584868  1.599737  38.766114  0.674284   
10         0.980228  0.212673  1.584826  1.599563  41.368600  0.674291   
11         0.976302  0.212894  1.58483

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_79738/1822548511.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_79738/1822548511.py:15: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')


In [37]:
sns.pairplot(end_of_run_ind_pd)
# hmm seems like one of the fits is comparatively bad and is an outlier

In [27]:
sns.pairplot(early_run_ind_pd)